<a href="https://colab.research.google.com/github/jmalarcontech/practica-teachable-machine/blob/main/memoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificador de imágenes con Teachable Machine

Antonio Jesús Molina Alarcón
Inteligencia Artificial, 2º DAM
IES Zaidín-Vergeles

Fecha: 20 de octubre de 2026
Repositorio: https://github.com/jmalarcontech/practica-teachable-machine
Página publicada: https://jmalarcontech.github.io/practica-teachable-machine/clasificador/


## 1. Introducción

El objetivo de esta práctica es entrenar un modelo de clasificación de imágenes con Teachable Machine y utilizarlo después en una página web propia. En mi caso, el modelo distingue entre un smartphone y un cargador.

El trabajo tiene tres partes: preparar las imágenes y entrenar el modelo, exportarlo a TensorFlow.js para ejecutarlo en local y, por último, convertir la página que genera Teachable Machine en una aplicación completa. Además de las siete mejoras que pide el enunciado (diseño, diseño adaptable, botones de inicio y parada, mensajes personalizados, umbral de coincidencia, histórico y subida de imágenes), la aplicación muestra en qué zona de la imagen se fija el modelo, elige el motor de cálculo más rápido del equipo y permite comprobar que sus resultados coinciden con los de la librería oficial.


## 2. Teachable Machine

Teachable Machine es una herramienta web de Google para entrenar modelos de clasificación sin programar. En los proyectos de imagen no entrena una red neuronal desde cero: parte de MobileNet, una red ya entrenada con más de un millón de fotografías, y solo ajusta las últimas capas con las imágenes que le damos. Esta técnica se conoce como aprendizaje por transferencia.

La ventaja es que MobileNet ya sabe reconocer bordes, formas y texturas, de modo que el modelo solo tiene que aprender qué combinación de esos rasgos corresponde a cada una de mis clases. Por eso basta con unos cientos de imágenes y el entrenamiento dura unos segundos. Todo el proceso ocurre en el navegador, sin enviar las imágenes a ningún servidor.


## 3. Imágenes de entrenamiento

### 3.1 Clases

He creado un proyecto de imagen estándar con tres clases: Smartphone, Cargador y Fondo.

El enunciado pide como mínimo dos. Añadí Fondo, con imágenes sin ningún objeto, porque el modelo siempre reparte el 100 % de la probabilidad entre las clases que conoce. Con solo dos clases, ante una pared vacía tendría que responder que ve un smartphone o un cargador; con la tercera puede indicar que no hay ninguno de los dos.

### 3.2 Toma de muestras

Grabé todas las muestras con la webcam, manteniendo pulsado *Hold to record* mientras movía el objeto para que cada fotograma fuese distinto.

- **Smartphone:** con la pantalla apagada y encendida, a varias distancias de la cámara.
- **Cargador:** con el enchufe abierto y cerrado (se puede plegar), también a varias distancias.
- **Fondo:** mi mano vacía, en la misma postura con la que sujeto los objetos, y los fondos solos, sin nada delante.

En las tres clases sostuve el objeto con la mano. Así la mano no sirve como pista para distinguir las clases y el modelo tiene que fijarse en el objeto. Por la misma razón grabé las tres clases delante de los mismos tres fondos: una cortina estampada, una pared blanca y mi ropa ocupando toda la imagen. Si un fondo apareciera solo en una clase, el modelo podría asociar ese fondo con esa clase y acertar por un motivo equivocado. El mapa de atención de la aplicación (apartado 8.10) sirve precisamente para comprobarlo.

En una primera versión, Fondo tenía muchas menos imágenes que las otras dos clases. Añadí más para equilibrarlas, porque con tan pocas el modelo tiende a ver un objeto donde no lo hay.

| Clase | Muestras (aproximadas) | Reservadas para la prueba |
|---|---|---|
| Smartphone | 906 | 136 |
| Cargador | 868 | 131 |
| Fondo | unas 560 | 84 |

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/clases.jpg)

*Figura 1. Las tres clases del proyecto, el panel de entrenamiento y el panel Under the hood.*

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/muestras_smartphone.jpg)

*Figura 2. Grabación de muestras de la clase Smartphone.*

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/muestras_cargador.jpg)

*Figura 3. Grabación de muestras de la clase Cargador.*


## 4. Entrenamiento

En el apartado *Advanced* se pueden ajustar tres parámetros:

- Épocas: cuántas veces recorre el modelo todas las imágenes.
- Tamaño de lote (*batch size*): cuántas imágenes procesa antes de corregir los pesos.
- Tasa de aprendizaje (*learning rate*): cuánto corrige los pesos en cada paso.

Probé con 50 y con 100 épocas, manteniendo el tamaño de lote en 16 y la tasa de aprendizaje en 0,001. El modelo final es el de 100 épocas.

| Parámetro | Valor |
|---|---|
| Épocas | 100 |
| Tamaño de lote | 16 |
| Tasa de aprendizaje | 0,001 |

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/parametros_50.jpg) ![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/parametros.jpg)

*Figura 4. Parámetros de los dos entrenamientos: 50 épocas (izquierda) y 100 épocas, el final (derecha).*

En la gráfica de precisión por época (figura 1, a la derecha), la precisión de entrenamiento y la de prueba llegan casi al 100 % en las primeras épocas y se mantienen. Con unas 2300 imágenes y un modelo que parte de MobileNet ya entrenado, aprender las tres clases le cuesta muy poco. Más épocas no mejoran el resultado, pero tampoco lo empeoran: no hay señales de que el modelo memorice en lugar de aprender.


## 5. Evaluación

Teachable Machine reserva automáticamente el 15 % de las imágenes, que el modelo no ve durante el entrenamiento, para evaluarlo.

| Clase | Precisión | Imágenes de prueba |
|---|---|---|
| Smartphone | 1,00 | 136 |
| Cargador | 0,99 | 131 |
| Fondo | 1,00 | 84 |

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/precision.jpg)

*Figura 5. Precisión por clase.*

En la matriz de confusión cada fila es la clase real y cada columna la que predice el modelo, así que los aciertos quedan en la diagonal. De 351 imágenes de prueba, el modelo solo falla una: un cargador que clasifica como smartphone. Es el error más esperable, porque los dos son objetos oscuros sujetos con la mano y vistos desde distancias parecidas.

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/confusion.jpg)

*Figura 6. Matriz de confusión.*

Un 99,7 % de acierto hay que interpretarlo con cuidado. Las imágenes de prueba salen de las mismas grabaciones que las de entrenamiento, así que se parecen muchísimo a ellas. La prueba de verdad es usar el modelo con otra luz, otro fondo u otra cámara, que es lo que permite la aplicación. Por eso grabé las muestras delante de tres fondos distintos y añadí a la aplicación el umbral de coincidencia: si en una situación nueva el modelo no está seguro, lo dice en lugar de equivocarse.


## 6. Exportación

Exporté el modelo en los dos formatos que pide la práctica. Para la página web usé la opción TensorFlow.js, que descarga un archivo comprimido con tres ficheros:

- `model.json`: la estructura de la red.
- `weights.bin`: los pesos aprendidos durante el entrenamiento.
- `metadata.json`: los nombres de las clases y el tamaño de imagen que espera el modelo.

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/exportacion.jpg)

*Figura 7. Ventana de exportación a TensorFlow.js.*

El formato TensorFlow (Keras) lo guardé para la práctica posterior. Además, guardé el proyecto en Google Drive para poder añadir imágenes y volver a entrenar sin empezar de cero.


## 7. Ejecución en local

Los ficheros del modelo van en la carpeta `modelo/`, con la estructura que indica el enunciado:

```
clasificador/
├── index.html
├── styles.css
├── app.js
└── modelo/
    ├── model.json
    ├── metadata.json
    └── weights.bin
```

Si se abre `index.html` con doble clic, la dirección empieza por `file://` y el navegador no deja leer `model.json`. Hay que servir la carpeta con un servidor local. Basta con el que incluye Python, ejecutado desde la carpeta del proyecto:

```bash
python3 -m http.server 8000
```

Después se abre `http://localhost:8000/clasificador/`. La extensión Live Server de Visual Studio Code hace lo mismo. Si aun así alguien abre la página con doble clic, la aplicación lo detecta, explica cómo solucionarlo y ofrece cargar los archivos del modelo a mano.

Primero probé el código que genera Teachable Machine sin cambios, salvo la ruta del modelo. Está en la carpeta `version-base/` del repositorio. Carga el modelo con `tmImage.load()`, abre la cámara con `tmImage.Webcam` y, en un bucle con `requestAnimationFrame`, captura un fotograma, llama a `model.predict()` y escribe la probabilidad de cada clase.


## 8. La aplicación


### 8.1 TensorFlow.js 4 en lugar de la librería de Teachable Machine

El código que exporta Teachable Machine usa la librería `@teachablemachine/image`, que depende de TensorFlow.js 1.3.1, una versión de 2019. En la aplicación cargo el modelo directamente con TensorFlow.js 4.22, la versión estable actual. Esto tiene tres ventajas:

- Puedo usar los motores de cálculo modernos (apartado 8.11).
- Tengo acceso a las capas intermedias del modelo, necesarias para el mapa de atención (apartado 8.10).
- No dependo de una librería que ya no se actualiza.

La condición es que el preprocesado sea idéntico al de Teachable Machine, porque el modelo solo funciona bien con imágenes preparadas igual que las de entrenamiento. La librería hace dos pasos, y la aplicación los replica tal cual: recortar el cuadrado central de la imagen y reducirlo a 224 × 224 píxeles, y pasar los valores de los píxeles de 0..255 a -1..1 dividiendo entre 127.

```javascript
// Igual que cropTo() de @teachablemachine/image: escala la imagen para que su lado
// menor mida "lado" píxeles, se queda con el cuadrado central y, si se pide, lo voltea.
function recortarCentro(origen, lado, espejo, lienzo) {
  const ancho = origen.videoWidth || origen.naturalWidth || origen.width;
  const alto = origen.videoHeight || origen.naturalHeight || origen.height;
  const escala = lado / Math.min(ancho, alto);
  // ...
}

// Misma normalización que Teachable Machine: de 0..255 a -1..1 dividiendo entre 127
x = tf.expandDims(tf.sub(tf.div(x, 127), 1), 0);
```

Para demostrar que el resultado es el mismo, el repositorio incluye la página `comprobacion/`. Carga el modelo dos veces, una con la librería oficial y otra con el código de la aplicación, clasifica las mismas fotos con las dos y compara las probabilidades.

En las pruebas, con modelos de la misma estructura que el mío, la diferencia máxima entre las probabilidades de las dos versiones fue del orden de 10⁻⁷. Esa diferencia se debe a la precisión numérica de cada motor de cálculo, no al preprocesado.


### 8.2 Organización del código

Respeté la estructura de archivos del enunciado: todo el código está en `index.html`, `styles.css` y `app.js`, sin librerías aparte de TensorFlow.js. El archivo `app.js` está dividido en secciones numeradas: configuración, utilidades, preprocesado, modelo, motores de cálculo, estado, carga del modelo, visor, cámara, imágenes, resultado, registro, cronograma, histórico, ajustes, diálogos, eventos e inicio.

La pieza central es la clase `Clasificador`, que carga el modelo, prepara cada imagen y devuelve las probabilidades y el mapa de atención. El resto del código se encarga de la interfaz.


### 8.3 Diseño

La página original solo muestra la cámara y una lista de textos con decimales. En la nueva versión la cámara ocupa un visor oscuro y, a su lado, el resultado se muestra en grande. Debajo aparecen la confianza, una barra de probabilidad por clase y la evolución de los últimos segundos.

Usé una sola tipografía, Archivo, que tiene un eje de anchura variable. La anchura marca la jerarquía: cuando hay un resultado, el nombre de la clase aparece en su versión más ancha; los estados sin resultado, como «En espera» o «No concluyente», usan una versión estrecha y en gris. Cada clase tiene un color propio, el mismo en las barras, la gráfica, el histórico y el mapa de atención. Los colores se comprobaron para que se distingan también con daltonismo, y Fondo va en gris porque significa que no hay objeto.

La página tiene tema claro y oscuro: sigue al sistema o se fuerza desde los ajustes. Los colores están definidos como variables CSS:

```css
:root {
  --pagina: #eceef1;
  --tinta: #1c2127;
  --serie-1: #2a78d6;
  --serie-2: #eb6834;
  --serie-fondo: #8a929c;
}
```

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/app_smartphone.jpg)

*Figura 8. La aplicación en escritorio reconociendo el smartphone con la cámara.*


### 8.4 Diseño adaptable

En el móvil todo se muestra en una sola columna, y en pantallas anchas el visor y el resultado se colocan uno junto a otro con CSS Grid. El tamaño del nombre de la clase se calcula para que la clase más larga quepa siempre en su columna. En pantallas muy estrechas se ocultan las columnas menos importantes del histórico, y los diálogos se abren como paneles desde la parte inferior.

En el móvil la aplicación abre por defecto la cámara trasera, que es la más cómoda para enfocar objetos, y solo aplica el efecto espejo a la cámara frontal.

```css
@media (min-width: 960px) {
  .mesa {
    grid-template-columns: minmax(0, 1.12fr) minmax(0, 1fr);
  }
}
```

<img src="https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/movil_smartphone.jpg" width="300">

*Figura 9. La aplicación en el móvil.*


<img src="https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/movil_resultado.jpg" width="300">

*Figura 10. Resultado en el móvil.*


### 8.5 Botones de inicio y parada

*Iniciar cámara* abre la cámara y arranca el análisis. *Detener cámara* lo para y apaga la cámara de verdad: se libera el dispositivo y se apaga su luz. Los botones se activan y desactivan según el estado, y la cámara también se detiene sola si se cambia de pestaña. Con la barra espaciadora se inicia o se detiene sin usar el ratón.

Para analizar la imagen uso `requestVideoFrameCallback`, que avisa cada vez que la cámara entrega un fotograma nuevo. Así nunca se analiza dos veces el mismo fotograma.

```javascript
function detenerCamara() {
  const estabaActiva = estado.camara.activa;
  estado.camara.activa = false;
  // ...
  if (estado.camara.stream) {
    estado.camara.stream.getTracks().forEach((pista) => pista.stop());
    estado.camara.stream = null;
  }
  ui.video.srcObject = null;
  actualizarDisponibilidad();
}
```


### 8.6 Mensaje personalizado

Bajo el nombre de la clase aparece un mensaje propio para cada una. Los mensajes se pueden cambiar desde *Ajustes*, se guardan en el navegador y, si se deja un campo vacío, se usa el mensaje por defecto. También se puede personalizar el mensaje de los resultados no concluyentes.

```javascript
function mensajePara(etiqueta) {
  const propio = estado.prefs.mensajes[etiqueta];
  return propio && propio.trim() ? propio.trim() : mensajePorDefecto(etiqueta);
}
```

<img src="https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/ajustes_mensajes.jpg" width="420">

*Figura 11. Ajustes de detección y mensajes personalizados de cada clase.*


### 8.7 Umbral de coincidencia

El control *Umbral* fija el porcentaje mínimo, entre el 50 % y el 99 %, para dar un resultado por bueno. Si la clase más probable no lo alcanza, la aplicación muestra «No concluyente» y explica cuál era la opción más probable.

El control comparte escala con las barras de probabilidad: su pulgar está exactamente a la altura de la marca vertical que aparece en cada barra, de modo que se ve de un vistazo qué clases superan el umbral. La gráfica de los últimos segundos dibuja el umbral como una línea discontinua.

```javascript
// Se compara con el valor redondeado que se muestra, para que pantalla y decisión coincidan
const aceptado = Math.round(confianza * 1000) / 10 >= estado.prefs.umbral;
```

Por ejemplo, con el umbral en el 80 %, si el modelo da un 65 % a Smartphone y un 35 % a Cargador, el resultado se marca como no concluyente.


### 8.8 Histórico de resultados

Cada resultado se guarda con una miniatura, la fecha, el origen, la clase, la confianza, el umbral aplicado y la probabilidad de todas las clases. El histórico se guarda en el navegador, se conserva al recargar la página, se puede filtrar por clase y se exporta a CSV (para abrirlo en Excel) o a JSON. Al pulsar un registro se abre su detalle, con el mapa de atención de ese momento.

Con la cámara había que tener en cuenta que el análisis se repite muchas veces por segundo. Si se guardaran todos los resultados, la tabla se llenaría de repetidos en un instante. Por eso un resultado solo se guarda cuando cambia y se mantiene un tiempo, 0,8 segundos por defecto. También se puede guardar un fotograma a mano con *Registrar fotograma*.

```javascript
if (ahora - estado.candidato.desde >= estado.prefs.estabilidad && clave !== estado.ultimaRegistrada) {
  estado.ultimaRegistrada = clave;
  registrarFotograma(false);
}
```

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/app_historico.jpg)

*Figura 12. Histórico de resultados en escritorio.*


<img src="https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/movil_historico.jpg" width="300">

*Figura 13. Histórico en el móvil.*


### 8.9 Clasificar imágenes subidas

En la pestaña *Imagen* se pueden elegir una o varias fotos, arrastrarlas o pegarlas desde el portapapeles. Cada foto se analiza con el mismo método que la cámara y se añade al histórico; si son varias, aparecen como miniaturas para revisarlas una a una.

El visor muestra la foto completa y oscurece lo que queda fuera del cuadrado central, que es la parte que analiza el modelo. Así se entiende por qué un objeto pegado al borde de una foto apaisada puede no detectarse.

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/app_imagen.jpg)

*Figura 14. Clasificación de una foto subida.*


### 8.10 Mapa de atención (Grad-CAM)

Además de decir qué ve, la aplicación muestra en qué parte de la imagen se ha fijado el modelo. Para ello usa Grad-CAM, una técnica de explicabilidad muy utilizada con redes convolucionales.

La idea es esta: MobileNet no entrega directamente una clase, sino 1280 «rasgos» para cada una de las 7 × 7 zonas en que divide la imagen. Grad-CAM calcula cuánto cambiaría la puntuación de la clase ganadora si aumentara cada rasgo (eso es el gradiente), da más peso a los rasgos que más influyen y suma, zona por zona. El resultado es una cuadrícula de 7 × 7 que la aplicación suaviza y dibuja sobre la imagen con el color de la clase.

```javascript
const gradiente = tf.grad((r) => tf.sum(tf.max(this.puntuar(r), 1)))(rasgos);
const pesos = tf.mean(gradiente, [1, 2], true);
const bruto = tf.relu(tf.sum(tf.mul(rasgos, pesos), 3));
```

Un detalle importante es que el gradiente se calcula sobre la puntuación previa al softmax: cuando el modelo está muy seguro, el softmax se satura y sus gradientes casi desaparecen, y el mapa saldría vacío.

El mapa tiene una utilidad práctica: si la zona resaltada es el objeto, el modelo ha aprendido lo que debía; si resalta la mano o la mesa, ha aprendido una pista falsa del entrenamiento (apartado 3.2). En la clase Fondo el mapa suele estar vacío, porque esa clase se reconoce precisamente por la ausencia de objetos.

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/app_cargador.jpg)

*Figura 15. Mapa de atención sobre el cargador.*


![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/app_fondo.jpg)

*Figura 16. Clase Fondo: sin objeto, el mapa queda vacío.*


<img src="https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/movil_cargador.jpg" width="300">

*Figura 17. Cargador reconocido desde el móvil.*


### 8.11 Motores de cálculo y rendimiento

TensorFlow.js puede calcular con la tarjeta gráfica (WebGL o WebGPU), con WebAssembly o con la CPU. Al cargar el modelo, la aplicación prueba en este orden: WebGL, WebAssembly y CPU. WebAssembly va antes que la CPU porque, en equipos con la tarjeta gráfica bloqueada, es muchas veces más rápido. El motor se puede cambiar en *Ajustes*, y si el elegido falla, la aplicación pasa al siguiente sin romperse.

```javascript
const orden = [...new Set([preferido === 'auto' ? null : preferido, 'webgl', 'wasm', 'cpu'].filter(Boolean))];
```

Bajo la gráfica se muestran el motor en uso, el tiempo de cada inferencia y los fotogramas analizados por segundo. La ficha del modelo (botón *Modelo*) muestra sus clases, parámetros, tamaño de los pesos, fecha de entrenamiento y tiempo de carga.

Para no agotar la memoria, todos los cálculos se hacen dentro de `tf.tidy()`, que libera automáticamente los tensores intermedios. Dejando la cámara en marcha, el número de tensores en memoria se mantiene constante.

<img src="https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/ajustes_rendimiento.jpg" width="420">

*Figura 18. Ajustes de cámara, motor de cálculo, apariencia y atajos de teclado.*


### 8.12 Otros detalles

- **Suavizado.** Con la cámara, las probabilidades se promedian con una media móvil exponencial, para que el resultado no parpadee entre fotogramas. La fórmula usa el tiempo transcurrido, así que funciona igual a 15 que a 60 fotogramas por segundo:

```javascript
const a = 1 - Math.exp(-dt / TAU_SUAVIZADO);
```

- **Cargar otro modelo.** Desde la ficha del modelo se puede cargar directamente el archivo .zip que descarga Teachable Machine, o sus tres archivos sueltos, sin copiarlos al proyecto. La aplicación descomprime el .zip en el propio navegador.
- **Accesibilidad.** Todo se puede usar con el teclado, los resultados se anuncian a los lectores de pantalla y la página respeta la preferencia de reducir animaciones. La página se ha revisado con la herramienta axe y no presenta errores según las pautas WCAG 2.2 AA.
- **Privacidad.** Todo el cálculo ocurre en el navegador: ninguna imagen sale del dispositivo.

<img src="https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/ficha_modelo.jpg" width="420">

*Figura 19. Ficha del modelo cargado.*


## 9. Publicación en GitHub

Subí el proyecto a un repositorio de GitHub con esta estructura:

```
practica-teachable-machine/
├── README.md
├── memoria.ipynb
├── index.html          (redirige al clasificador)
├── clasificador/       (la aplicación)
├── comprobacion/       (comparación con la librería oficial)
└── version-base/       (código original de Teachable Machine)
```

También activé GitHub Pages (en *Settings*, apartado *Pages*, publicando desde la rama `main`), de forma que la aplicación se puede usar desde cualquier dispositivo sin instalar nada. Como GitHub Pages sirve las páginas por HTTPS, el navegador permite usar la cámara, algo que no ocurre en una dirección `http` normal salvo en `localhost`.

![](https://raw.githubusercontent.com/jmalarcontech/practica-teachable-machine/main/capturas/github_pages_ajustes.jpg)

*Figura 20. Configuración de GitHub Pages.*


## 10. Problemas y soluciones

- Al abrir `index.html` con doble clic el modelo no carga, porque el navegador bloquea las peticiones desde `file://`. Se soluciona con un servidor local, y la aplicación explica el problema y permite cargar el modelo a mano.
- Por el mismo motivo, el navegador no ejecuta los módulos de JavaScript (`type="module"`) abiertos con doble clic. Por eso `app.js` se carga como un script clásico.
- TensorFlow.js rechaza WebGL en equipos con la tarjeta gráfica bloqueada y pasa a la CPU, que es muy lenta. La aplicación prueba antes WebAssembly.
- Algunos navegadores tienen la API de WebGPU pero ninguna tarjeta compatible. Antes de ofrecer ese motor, la aplicación comprueba que exista un adaptador.
- Con la cámara, el histórico se llenaría de registros repetidos. Se resuelve guardando solo los resultados que se mantienen estables.
- Si el modelo está muy seguro, el mapa de atención calculado sobre el softmax sale vacío. Se calcula sobre la puntuación previa.
- Al descargar dos veces el modelo, el navegador renombra el archivo a `weights (1).bin`. Al cargar archivos sueltos, la aplicación no depende del nombre exacto.
- En iPhone, Safari solo reproduce la cámara dentro de la página si el vídeo tiene el atributo `playsinline`.
- En la primera versión del modelo, la clase Fondo tenía muchas menos imágenes que las demás. Se equilibró grabando más muestras de la mano vacía y de los fondos solos.


## 11. Conclusiones

El aprendizaje por transferencia permite conseguir un modelo útil con pocas imágenes y en muy poco tiempo: en las primeras épocas el modelo ya acierta casi todas las imágenes de prueba. Lo que más influye en el resultado no son los parámetros de entrenamiento, sino cómo se toman las muestras: variar distancias, posturas y fondos, y grabar todas las clases en las mismas condiciones.

La clase Fondo y el umbral de coincidencia hacen que la aplicación sea más fiable en un uso real: cuando el modelo no tiene claro lo que ve, lo indica en lugar de equivocarse. El mapa de atención añade algo que una simple probabilidad no da: permite ver si el modelo acierta por la razón correcta.

El punto débil previsible es la distinción entre el cargador y el smartphone, que es el único error de la evaluación. Para mejorarlo grabaría más muestras con el cargador en posiciones en las que se parece menos a un móvil y con fondos e iluminaciones nuevos, como la del aula.


## 12. Guarda y exporta

- Proyecto guardado en Google Drive, para poder añadir muestras y volver a entrenar.
- Modelo exportado a TensorFlow.js, en la carpeta `clasificador/modelo/` del repositorio.
- Modelo exportado también a TensorFlow (Keras), en la carpeta [`modelo-keras/`](https://github.com/jmalarcontech/practica-teachable-machine/tree/main/modelo-keras) del repositorio (`keras_model.h5` y `labels.txt`), para la práctica siguiente.


## Anexo: código fuente

El código completo está en el repositorio. Cada archivo se puede abrir desde estos enlaces:

| Archivo | Contenido |
|---|---|
| [clasificador/index.html](https://github.com/jmalarcontech/practica-teachable-machine/blob/main/clasificador/index.html) | Estructura de la página: visor, controles, resultado, histórico y diálogos de ajustes, modelo y detalle |
| [clasificador/styles.css](https://github.com/jmalarcontech/practica-teachable-machine/blob/main/clasificador/styles.css) | Diseño, temas claro y oscuro y reglas para móvil |
| [clasificador/app.js](https://github.com/jmalarcontech/practica-teachable-machine/blob/main/clasificador/app.js) | Carga del modelo, preprocesado, inferencia, mapa de atención, cámara, imágenes, histórico y ajustes |
| [clasificador/modelo/](https://github.com/jmalarcontech/practica-teachable-machine/blob/main/clasificador/modelo) | Modelo exportado de Teachable Machine: model.json, metadata.json y weights.bin |
| [comprobacion/](https://github.com/jmalarcontech/practica-teachable-machine/blob/main/comprobacion) | Comparación de resultados con la librería oficial de Teachable Machine |
| [version-base/index.html](https://github.com/jmalarcontech/practica-teachable-machine/blob/main/version-base/index.html) | Código original generado por Teachable Machine |

El archivo `app.js` está dividido en secciones numeradas, y cada función lleva un comentario que explica qué hace. Los fragmentos más importantes están explicados en el apartado 8.
